In [19]:
import os
import shutil
import pandas as pd
from glob import glob
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from skimage import io
import math
from PIL import Image, ImageOps, ImageFilter

# 4.3  Evaluation of classification pipeline on historical dataset

This notebook descrebes the final state of the proposed data processing pipeline in a scope of an environmental research.

In [2]:
from common.configuration import ModelPath, DatasetPath, species_invasive, species_native
from pipeline.full_predict import ImageProcessor
from common.configuration import DatasetPath, ModelPath, species_native, species_label_map_all, species_classes_map_all

In [3]:
model_class_to_c_number_remap = {}
for i, c in enumerate(species_native):
    model_class_to_c_number_remap[i] = species_classes_map_all[c]

def prediction_to_class_name(prediction):
    predicted_c_number = model_class_to_c_number_remap.get(prediction, -1)
    return species_label_map_all[predicted_c_number]

In [4]:
image_processor = ImageProcessor(ModelPath.instance_segmentation, ModelPath.osc_classification, prediction_to_class_name)

### Reference samples dataset

Load test subset of full images


In [5]:
all_images = glob(os.path.join(DatasetPath.real_historical_samples_classification, "*", "*.png"))

In [17]:
images_ds = []
for img in all_images:
    images_ds.append(
        {
            "full_path" : img,
            "file_name" : os.path.split(img)[1],
            "microscopy_name" : os.path.split(img)[1].rsplit("_", 1)[0],
            "label"  : os.path.split(os.path.split(img)[0])[1]
        }
    )
images_ds = pd.DataFrame(images_ds)
images_ds

,full_path,file_name,microscopy_name,label
0,Dataset/pollen_dataset_2025_02_05\historical_c...,01 jul (152)_0.png,01 jul (152),0_Not pollen
1,Dataset/pollen_dataset_2025_02_05\historical_c...,01 jul (210)_0.png,01 jul (210),0_Not pollen
2,Dataset/pollen_dataset_2025_02_05\historical_c...,01 jul (230)_0.png,01 jul (230),0_Not pollen
3,Dataset/pollen_dataset_2025_02_05\historical_c...,01 jul (231)_0.png,01 jul (231),0_Not pollen
4,Dataset/pollen_dataset_2025_02_05\historical_c...,01 jul (303)_0.png,01 jul (303),0_Not pollen
...,...,...,...,...
5725,Dataset/pollen_dataset_2025_02_05\historical_c...,september 2 (27)_0.png,september 2 (27),Urtica
5726,Dataset/pollen_dataset_2025_02_05\historical_c...,september 2 (28)_0.png,september 2 (28),Urtica
5727,Dataset/pollen_dataset_2025_02_05\historical_c...,september 2 (29)_0.png,september 2 (29),Urtica
5728,Dataset/pollen_dataset_2025_02_05\historical_c...,september 2 (41)_0.png,september 2 (41),Urtica


### Process all images

### Perform predictions

In [18]:
predictions_path = "Save/historical_classification_predictions.csv"

In [26]:
predictions = []
for ind, item in tqdm(images_ds.iterrows()):
    pred = item.to_dict()
    with Image.open(item.full_path) as obj:
        pred_class, embeddings = image_processor._classify_image(
                obj, augment=0, threshold=8.0)
        pred["pred_class"] = pred_class
        pred["embeddings"] = embeddings
        predictions.append(pred)

predictions_df = pd.DataFrame(predictions)
predictions_df.to_csv(predictions_path)

0it [00:00, ?it/s]